# 🎬 What Makes a Movie Successful?
### Exploratory Data Analysis — TMDB Movie Dataset

**Goal:** Uncover patterns in movie budgets, revenues, genres, and ratings using real-world data.

**Dataset:** [TMDB Movie Metadata on Kaggle](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)  
Download `tmdb_5000_movies.csv` and place it in the same folder as this notebook.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ast
import warnings

warnings.filterwarnings('ignore')

# Chart style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Libraries loaded successfully ✓')

## 2. Load & First Look

In [ ]:
df = pd.read_csv('tmdb_5000_movies.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns:\n{list(df.columns)}')
df.head(3)

In [ ]:
# Data types and missing values at a glance
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'nulls': df.isnull().sum(),
    'null_%': (df.isnull().mean() * 100).round(1)
})
summary[summary['nulls'] > 0]

## 3. Data Cleaning

Key issues to fix:
- Replace zeros in `budget` and `revenue` with NaN (0 means data is missing, not free)
- Parse `release_date` as datetime
- Extract genre names from the nested JSON string in `genres`

In [ ]:
# --- Replace 0s with NaN for financial columns ---
df['budget'] = df['budget'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)

# --- Parse dates ---
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year
df['release_decade'] = (df['release_year'] // 10 * 10).astype('Int64')

# --- Parse genres from JSON string ---
def extract_names(json_str):
    """Extract 'name' values from a JSON list-of-dicts string."""
    try:
        items = ast.literal_eval(json_str)
        return [item['name'] for item in items]
    except:
        return []

df['genre_list'] = df['genres'].apply(extract_names)
df['primary_genre'] = df['genre_list'].apply(lambda x: x[0] if x else 'Unknown')
df['genre_count'] = df['genre_list'].apply(len)

# --- ROI (Return on Investment) ---
df['roi'] = ((df['revenue'] - df['budget']) / df['budget']) * 100

print('Cleaning complete ✓')
print(f"Movies with budget data: {df['budget'].notna().sum():,}")
print(f"Movies with revenue data: {df['revenue'].notna().sum():,}")
df[['title', 'release_year', 'budget', 'revenue', 'roi', 'genre_list', 'primary_genre']].head(5)

## 4. Univariate Analysis
### 4.1 Distribution of movie ratings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Vote average
axes[0].hist(df['vote_average'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of ratings')
axes[0].set_xlabel('Vote average (0–10)')
axes[0].axvline(df['vote_average'].median(), color='tomato', linestyle='--', label=f"Median: {df['vote_average'].median():.1f}")
axes[0].legend()

# Budget (log scale)
budget_data = df['budget'].dropna()
axes[1].hist(np.log10(budget_data), bins=30, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Distribution of budgets (log scale)')
axes[1].set_xlabel('log10(Budget in USD)')

# Runtime
runtime_data = df['runtime'].dropna()
runtime_data = runtime_data[(runtime_data > 30) & (runtime_data < 300)]
axes[2].hist(runtime_data, bins=40, color='mediumpurple', edgecolor='white')
axes[2].set_title('Distribution of runtime')
axes[2].set_xlabel('Runtime (minutes)')
axes[2].axvline(runtime_data.median(), color='tomato', linestyle='--', label=f"Median: {runtime_data.median():.0f} min")
axes[2].legend()

plt.suptitle('Movie attribute distributions', fontsize=13, fontweight='500', y=1.02)
plt.tight_layout()
plt.savefig('plot_distributions.png', bbox_inches='tight')
plt.show()

## 5. Correlation Analysis
### Does budget predict revenue?

In [ ]:
fin = df[['budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'runtime']].dropna(subset=['budget', 'revenue'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Budget vs Revenue
axes[0].scatter(np.log10(fin['budget']), np.log10(fin['revenue']),
                alpha=0.3, s=15, color='steelblue')
corr = fin[['budget', 'revenue']].corr().iloc[0, 1]
axes[0].set_title(f'Budget vs Revenue  (r = {corr:.2f})')
axes[0].set_xlabel('log10(Budget)')
axes[0].set_ylabel('log10(Revenue)')

# Correlation heatmap
corr_matrix = fin.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1],
            linewidths=0.5, square=True)
axes[1].set_title('Correlation matrix')

plt.tight_layout()
plt.savefig('plot_correlations.png', bbox_inches='tight')
plt.show()

## 6. Genre Analysis
### Which genres dominate — by volume and by revenue?

In [ ]:
# Explode genre_list so each row = one movie-genre pair
genres_exploded = df.explode('genre_list').rename(columns={'genre_list': 'genre'})
genres_exploded = genres_exploded[genres_exploded['genre'].notna() & (genres_exploded['genre'] != '')]

# Count per genre
genre_counts = genres_exploded['genre'].value_counts().head(12)

# Median revenue per genre
genre_revenue = (genres_exploded.groupby('genre')['revenue']
                  .median()
                  .dropna()
                  .sort_values(ascending=False)
                  .head(12))

# Median rating per genre
genre_rating = (genres_exploded.groupby('genre')['vote_average']
                 .median()
                 .dropna()
                 .sort_values(ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Volume
genre_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Movies per genre')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

# Revenue
genre_revenue.plot(kind='barh', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Median revenue per genre')
axes[1].set_xlabel('Revenue (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
axes[1].invert_yaxis()

# Rating
genre_rating.plot(kind='barh', ax=axes[2], color='mediumpurple')
axes[2].set_title('Median rating per genre')
axes[2].set_xlabel('Vote average')
axes[2].invert_yaxis()

plt.suptitle('Genre breakdown', fontsize=13, fontweight='500', y=1.02)
plt.tight_layout()
plt.savefig('plot_genres.png', bbox_inches='tight')
plt.show()

## 7. Time Trends
### How have ratings and volumes changed over the decades?

In [ ]:
yearly = (df[df['release_year'] >= 1960]
          .groupby('release_year')
          .agg(
              movie_count=('title', 'count'),
              avg_rating=('vote_average', 'mean'),
              median_budget=('budget', 'median'),
              median_revenue=('revenue', 'median')
          ).reset_index())

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Number of movies per year
axes[0].fill_between(yearly['release_year'], yearly['movie_count'],
                     alpha=0.4, color='steelblue')
axes[0].plot(yearly['release_year'], yearly['movie_count'], color='steelblue', linewidth=1.5)
axes[0].set_title('Number of movies released per year')
axes[0].set_ylabel('Count')

# Average rating over time
axes[1].plot(yearly['release_year'], yearly['avg_rating'], color='tomato', linewidth=2)
axes[1].set_title('Average movie rating per year')
axes[1].set_ylabel('Avg vote average')
axes[1].set_xlabel('Year')
axes[1].set_ylim(4, 8)
axes[1].axhline(yearly['avg_rating'].mean(), color='gray', linestyle='--',
                label=f"Overall mean: {yearly['avg_rating'].mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_trends.png', bbox_inches='tight')
plt.show()

## 8. Key Insight Chart
### The tension: high-grossing genres vs critically acclaimed genres

In [ ]:
genre_summary = (genres_exploded
    .groupby('genre')
    .agg(
        count=('title', 'count'),
        median_revenue=('revenue', 'median'),
        median_rating=('vote_average', 'median')
    )
    .dropna()
    .reset_index())

# Keep genres with enough movies
genre_summary = genre_summary[genre_summary['count'] >= 50]

fig, ax = plt.subplots(figsize=(11, 7))

scatter = ax.scatter(
    genre_summary['median_revenue'] / 1e6,
    genre_summary['median_rating'],
    s=genre_summary['count'] * 2,
    alpha=0.6,
    c=genre_summary['count'],
    cmap='viridis',
    edgecolors='white',
    linewidths=0.5
)

# Label each bubble
for _, row in genre_summary.iterrows():
    ax.annotate(
        row['genre'],
        (row['median_revenue'] / 1e6, row['median_rating']),
        textcoords='offset points', xytext=(6, 4),
        fontsize=9, color='#333'
    )

plt.colorbar(scatter, label='Number of movies', ax=ax)

ax.set_xlabel('Median revenue (USD millions)', fontsize=11)
ax.set_ylabel('Median rating (0–10)', fontsize=11)
ax.set_title(
    'Revenue vs Rating by Genre\n'
    '(bubble size = number of movies)',
    fontsize=13, fontweight='500'
)

# Quadrant lines
ax.axvline(genre_summary['median_revenue'].median() / 1e6, color='gray', linestyle=':', alpha=0.6)
ax.axhline(genre_summary['median_rating'].median(), color='gray', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig('plot_key_insight.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n📌 Key insight:')
print('Top-right quadrant = High revenue AND high rating — the sweet spot.')
print('Which genres land there? Those are your blockbuster + critic-approved films.')

## 9. Summary & Findings

Fill this section in after running all cells. Write 3–5 bullet points summarising what you found. This is what goes in your GitHub README.

**Example structure:**
- 📊 The dataset contains X movies spanning Y years...
- 💰 Budget and revenue are moderately correlated (r = ...), but high budget does not guarantee success...
- 🎬 Drama is the most common genre, but Animation generates the highest median revenue...
- ⭐ Documentary and History genres receive the highest average ratings...
- 📅 Movie production increased sharply after...

---

**Next steps to extend this project:**
- Add a regression model to predict revenue from budget + genre + release year (connects to Phase 2)
- Build an interactive Streamlit dashboard from these charts (connects to Phase 4)
- Analyse cast/crew data from the companion `tmdb_5000_credits.csv` file